<a href="https://colab.research.google.com/github/andarahabibie/DataScienceToolsEcosystem/blob/main/LSB_UAS_Crypto_Stegano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

def get_rgb_from_int(pixel_value):
    """Mengurai nilai integer 24-bit menjadi komponen R, G, B."""
    r = (pixel_value >> 16) & 0xFF
    g = (pixel_value >> 8) & 0xFF
    b = pixel_value & 0xFF
    return r, g, b

def get_int_from_rgb(r, g, b):
    """Menggabungkan komponen R, G, B menjadi satu nilai integer 24-bit."""
    return (r << 16) + (g << 8) + b

def string_to_binary(s):
    """Mengubah string menjadi aliran bit biner berdasarkan kode ASCII."""
    return ''.join(format(ord(char), '08b') for char in s)

def embed_and_calculate_psnr(m, nim_string, original_pixels_int):
    """
    Menyisipkan pesan ke dalam piksel menggunakan m-LSB dan menghitung PSNR.
    """
    # Langkah 1: Persiapan Data
    message_bits = string_to_binary(nim_string)
    message_len = len(message_bits)

    original_components = []
    for p_int in original_pixels_int:
        original_components.extend(get_rgb_from_int(p_int))

    num_components = len(original_components)
    capacity = num_components * m

    if message_len > capacity:
        print(f"Untuk m={m}, Kapasitas tidak cukup. Butuh {message_len} bit, tersedia {capacity} bit.")
        return None, None

    # Langkah 2: Proses Penyisipan
    stego_components = list(original_components)
    message_idx = 0

    # Buat bitmask untuk membersihkan m-LSB
    # Contoh m=3: mask = ~( (1<<3) - 1 ) = ~(8-1) = ~7 = ~0b111 -> ...11111000
    mask = ~((1 << m) - 1)

    for i in range(num_components):
        if message_idx < message_len:
            # Ambil m bit dari pesan
            bits_to_embed_str = message_bits[message_idx : message_idx + m]

            # Jika di akhir pesan dan bit tidak cukup m, tambahkan padding 0
            bits_to_embed_str = bits_to_embed_str.ljust(m, '0')
            bits_to_embed = int(bits_to_embed_str, 2)

            # Bersihkan m-LSB dari komponen piksel asli
            cleared_component = stego_components[i] & mask

            # Gabungkan dengan bit pesan
            stego_components[i] = cleared_component | bits_to_embed

            message_idx += m
        else:
            break

    # Langkah 3: Menghitung MSE dan PSNR
    mse_sum = 0
    for i in range(len(original_components)):
        error = original_components[i] - stego_components[i]
        mse_sum += error * error

    mse = mse_sum / len(original_components)

    if mse == 0:
        return float('inf'), stego_components

    max_i = 255.0
    psnr = 10 * math.log10( (max_i**2) / mse )

    return psnr, stego_components

# --- Data Awal ---
original_pixels = [
    414030, 1387162, 2370685, 5425979, 962226,
    2451490, 991205, 4382701, 729657, 8429951,
    28412, 7401231, 3341202, 11923841, 3700931,
    489091, 7183923, 3593, 10034321, 12000012
]

# GANTI DENGAN 5 DIGIT TERAKHIR NIM 2211502014(02014)
my_nim = "02014"

print(f"NIM yang akan disisipkan: {my_nim}")
print(f"Pesan biner (40 bit): {string_to_binary(my_nim)}\n")
print("-" * 50)
print("Mencari nilai m maksimal dengan PSNR >= 30 dB...")
print("-" * 50)

max_tolerable_m = 0
final_psnr = 0

for m_val in range(1, 9): # Coba untuk m dari 1 sampai 8
    psnr_result, stego_data = embed_and_calculate_psnr(m_val, my_nim, original_pixels)

    if psnr_result is not None:
        print(f"Hasil untuk m = {m_val}:")
        print(f"  -> PSNR = {psnr_result:.2f} dB")
        if psnr_result >= 30.0:
            print("  -> Status: Toleransi DITERIMA\n")
            max_tolerable_m = m_val
            final_psnr = psnr_result
        else:
            print("  -> Status: Toleransi DITOLAK\n")
            # Berhenti setelah menemukan m yang tidak dapat ditoleransi
            break
    else:
        # Jika kapasitas tidak cukup, hentikan loop
        break

print("-" * 50)
print("KESIMPULAN")
print("-" * 50)
if max_tolerable_m > 0:
    print(f"NIM dapat disisipkan hingga M = {max_tolerable_m}.")
    print(f"Pada m = {max_tolerable_m}, nilai PSNR yang didapat adalah {final_psnr:.2f} dB.")
else:
    print("Tidak ditemukan nilai m yang memenuhi kriteria PSNR >= 30 dB.")

print("\nPerhitungan selesai.")

NIM yang akan disisipkan: 02014
Pesan biner (40 bit): 0011000000110010001100000011000100110100

--------------------------------------------------
Mencari nilai m maksimal dengan PSNR >= 30 dB...
--------------------------------------------------
Hasil untuk m = 1:
  -> PSNR = 54.45 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 2:
  -> PSNR = 47.78 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 3:
  -> PSNR = 43.56 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 4:
  -> PSNR = 38.79 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 5:
  -> PSNR = 35.42 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 6:
  -> PSNR = 32.21 dB
  -> Status: Toleransi DITERIMA

Hasil untuk m = 7:
  -> PSNR = 26.15 dB
  -> Status: Toleransi DITOLAK

--------------------------------------------------
KESIMPULAN
--------------------------------------------------
NIM dapat disisipkan hingga M = 6.
Pada m = 6, nilai PSNR yang didapat adalah 32.21 dB.

Perhitungan selesai.
